In [1]:
!pip install langchain pypdf openai tiktoken langchain_community

  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
  Using cached pydantic-2.12.0-py3-none-any.whl.metadata (83 kB)
     ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
     --- ------------------------------------ 0.8/9.8 MB 3.1 MB/s eta 0:00:03
     ----- ---------------------------------- 1.3/9.8 MB 2.8 MB/s eta 0:00:04
     ------- -------------------------------- 1.8/9.8 MB 2.7 MB/s eta 0:00:03
     --------- ------------------------------ 2.4/9.8 MB 2.7 MB/s eta 0:00:03
     ----------- ---------------------------- 2.9/9.8 MB 2.7 MB/s eta 0:00:03
     ------------- -------------------------- 3.4/9.8 MB 2.7 MB/s eta 0:00:03
     ---------------- ----------------------- 3.9/9.8 MB 2.6 MB/s eta 0:00:03
     ------------------- -------------------- 4.7/9.8 MB 2.6 MB/s eta 0:00:02
     --------------------- ------------------ 5.2/9.8 MB 2.6 MB/s eta 0:00:02
     --------


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Langchain dependencies
from langchain.document_loaders.pdf import PyPDFDirectoryLoader # Importing PDF loader from Langchain
from langchain.text_splitter import RecursiveCharacterTextSplitter # Importing text splitter from Langchain
from langchain.embeddings import OpenAIEmbeddings # Importing OpenAI embeddings from Langchain
from langchain.schema import Document # Importing Document schema from Langchain
from langchain_chroma import Chroma
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv # Importing dotenv to get API key from .env file
from langchain.chat_models import ChatOpenAI # Import OpenAI LLM
import os # Importing os module for operating system functionalities
import shutil # Importing shutil module for high-level file operations

In [2]:
# Directory to your pdf files:
DATA_PATH = "./data/uploaded_files"
def load_documents():
  """
  Load PDF documents from the specified directory using PyPDFDirectoryLoader.
  Returns:
  List of Document objects: Loaded PDF documents represented as Langchain
                                                          Document objects.
  """
  # Initialize PDF loader with specified directory
  document_loader = PyPDFDirectoryLoader(DATA_PATH) 
  # Load PDF documents and return them as a list of Document objects
  return document_loader.load() 

documents = load_documents() # Call the function
# Inspect the contents of the first document as well as metadata
print(documents[0])

page_content='CaseId: 202045001
Employer: Wide World Importers
Location: Miami 
Event Date: 1/1/2015
Event: Caught in or compressed by equipment or objects, unspeciﬁed 
Nature: multiple injuries
Final Narrative Hospitalized Amputation Part Of
Body Source
An employee's leg was pinned between a truck and
the powered pallet jack being operated. The
employee was hospitalized for treatment/surgery at
Navicent Health.
1 0 legs pallet jack-
powered
Page 1 of 1' metadata={'producer': 'cairo 1.15.10 (http://cairographics.org)', 'creator': 'PyPDF', 'creationdate': '2020-06-29T18:04:45+00:00', 'author': '', 'keywords': '', 'source': 'data\\uploaded_files\\202045001.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [3]:
def split_text(documents: list[Document]):
  """
  Split the text content of the given list of Document objects into smaller chunks.
  Args:
    documents (list[Document]): List of Document objects containing text content to split.
  Returns:
    list[Document]: List of Document objects representing the split text chunks.
  """
  # Initialize text splitter with specified parameters
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, # Size of each chunk in characters
    chunk_overlap=100, # Overlap between consecutive chunks
    length_function=len, # Function to compute the length of the text
    add_start_index=True, # Flag to add start index to each chunk
  )

  # Split documents into smaller chunks using text splitter
  chunks = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

  # Print example of page content and metadata for a chunk
  document = chunks[0]
  print(document.page_content)
  print(document.metadata)

  return chunks # Return the list of split text chunks

# Path to the directory to save Chroma database
CHROMA_PATH = "chroma"
def save_to_chroma(chunks: list[Document]):
  """
  Save the given list of Document objects to a Chroma database.
  Args:
  chunks (list[Document]): List of Document objects representing text chunks to save.
  Returns:
  None
  """

  # Clear out the existing database directory if it exists
  if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

  # Create a new Chroma database from the documents using OpenAI embeddings
  db = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(),
    persist_directory=CHROMA_PATH
  )

  # Persist the database to disk
  # db.persist()
  print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")

In [4]:
def generate_data_store():
  """
  Function to generate vector database in chroma from documents.
  """
  documents = load_documents() # Load documents from a source
  chunks = split_text(documents) # Split documents into manageable chunks
  save_to_chroma(chunks) # Save the processed data to a data store

# Load environment variables from a .env file
load_dotenv()
# Generate the data store
generate_data_store()

Split 1 documents into 2 chunks.
CaseId: 202045001
Employer: Wide World Importers
Location: Miami 
Event Date: 1/1/2015
Event: Caught in or compressed by equipment or objects, unspeciﬁed 
Nature: multiple injuries
Final Narrative Hospitalized Amputation Part Of
Body Source
An employee's leg was pinned between a truck and
{'producer': 'cairo 1.15.10 (http://cairographics.org)', 'creator': 'PyPDF', 'creationdate': '2020-06-29T18:04:45+00:00', 'author': '', 'keywords': '', 'source': 'data\\uploaded_files\\202045001.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'start_index': 0}


C:\Users\User\AppData\Local\Temp\ipykernel_4016\3175646688.py:46: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  OpenAIEmbeddings(),


Saved 2 chunks to chroma.


In [10]:
user_query = "Tell me the Employer name and its location along with the event details"

In [15]:
def query_rag(user_query):
  """
  Query a Retrieval-Augmented Generation (RAG) system using Chroma database and OpenAI.
  Args:
    - query_text (str): The text to query the RAG system with.
  Returns:
    - formatted_response (str): Formatted response including the generated text and sources.
    - response_text (str): The generated response text.
  """
  # YOU MUST - Use same embedding function as before
  embedding_function = OpenAIEmbeddings(model='text-embedding-3-small')

  # Prepare the database
  db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)
  
  # Retrieving the context from the DB using similarity search
  results = db.similarity_search_with_relevance_scores(user_query, k=3)

  # Check if there are any matching results or if the relevance score is too low
  if len(results) == 0:
    print(f"Unable to find matching results.")

  # Combine context from matching documents
  context_text = "\n\n - -\n\n".join([doc.page_content for doc, _score in results])

  PROMPT_TEMPLATE = "Please answer the following question based on the information provided below. If the answer is not found in the content, please reply with 'don't know'.\n\n{context}\n\nQuestion: {question}"
    
  # Create prompt template using context and query text
  prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
  prompt = prompt_template.format(context=context_text, question=query_text)
  
  # Initialize OpenAI chat model
  model = ChatOpenAI(model='gpt-4o-mini')

  # Generate response text based on the prompt
  response_text = model.predict(prompt)
 
   # Get sources of the matching documents
  sources = [doc.metadata.get("source", None) for doc, _score in results]
 
  # Format and return response including generated text and sources
  formatted_response = f"Response: {response_text}\nSources: {sources}"
  return formatted_response, response_text

# Let's call our function we have defined
formatted_response, response_text = query_rag(query_text)
# and finally, inspect our final response!
print(response_text)

C:\Users\User\AppData\Local\Temp\ipykernel_4016\2466315190.py:17: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='5725bd41-e52e-43d4-ae1d-884cc6a49438', metadata={'source': 'data\\uploaded_files\\202045001.pdf', 'start_index': 229, 'author': '', 'producer': 'cairo 1.15.10 (http://cairographics.org)', 'total_pages': 1, 'creator': 'PyPDF', 'keywords': '', 'creationdate': '2020-06-29T18:04:45+00:00', 'page': 0, 'page_label': '1'}, page_content="Body Source\nAn employee's leg was pinned between a truck and\nthe powered pallet jack being operated. The\nemployee was hospitalized for treatment/surgery at\nNavicent Health.\n1 0 legs pallet jack-\npowered\nPage 1 of 1"), -0.3923255238296295), (Document(id='5f415411-8304-4e59-a290-ec1bb5162e49', metadata={'total_pages': 1, 'start_index': 0, 'creator': 'PyPDF', 'source': 'data\\uploaded_files\\202045001.pdf', 'page_label': '1', 'creationdate': '2020-06-29T18:04:45+00:00', 'producer': 'cairo 1.15.10 (http://cairographics.

Employer: Wide World Importers  
Location: Miami  
Event Date: 1/1/2015  
Event: Caught in or compressed by equipment or objects, unspecified  
Nature: multiple injuries  


In [16]:
dir(db)

NameError: name 'db' is not defined

In [ ]:
from langchain_community.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from datetime import datetime
embeddings = OpenAIEmbeddings()
indexer_path = "./data/faiss_indexes/index.pkl"
# indexer_path = os.path.join(faiss_indexer_path,filename)
# if not os.path.exists(indexer_path):
#     return jsonify({"status":"File does not exists. Please provide correct filename"})
db = FAISS.load_local(indexer_path, embeddings)
embedding_vector = embeddings.embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector)
print(f"content: {docs[0].page_content}")

start = datetime.now()
answer = ask_gpt3(docs[0].page_content,query)
answer = answer.split("\n")[0]
answer = answer.replace("Answer:","").strip()
response = {"query":query,"answer":answer.split("\n")[0]}
end = datetime.now()
t2 = datetime.now()
print(f"time taken: {t2-t1}")
print(f"time taken by gpt3: {end-start}")

In [5]:
import langchain
import pandas as pd

In [6]:
import os
import getpass

os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key:')

OpenAI API Key:··········


In [7]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader

In [15]:
from langchain.document_loaders import TextLoader,PyPDFLoader

filepath = '/content/202045001.pdf'
filename = filepath.split("/")[-1]
loader = PyPDFLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(docs, embeddings)
db.save_local(filename)


In [18]:
db = FAISS.from_documents(docs, embeddings)
query = "What is the location of the employer Wide World Importers ?"
docs = db.similarity_search(query)

In [29]:
# docs_and_scores = db.similarity_search_with_score(query)
# docs_and_scores

In [23]:
embedding_vector = embeddings.embed_query(query)
docs_and_scores = db.similarity_search_by_vector(embedding_vector)
docs_and_scores

[Document(page_content="CaseId: 202045001\nEmployer: Wide World Importers\nLocation: Miami \nEvent Date: 1/1/2015\nEvent: Caught in or compressed by equipment or objects, unspeciﬁed \nNature: multiple injuries\nFinal Narrative Hospitalized AmputationPart Of\nBodySource\nAn employee's leg was pinned between a truck and\nthe powered pallet jack being operated. The\nemployee was hospitalized for treatment/surgery at\nNavicent Health.1 0 legs pallet jack-\npowered\nPage 1 of 1", metadata={'source': '/content/202045001.pdf', 'page': 0})]

In [24]:
db.save_local("faiss_index")

In [26]:
new_db = FAISS.load_local("faiss_index", embeddings)

In [27]:
docs = new_db.similarity_search(query)

In [31]:
docs[0].page_content

"CaseId: 202045001\nEmployer: Wide World Importers\nLocation: Miami \nEvent Date: 1/1/2015\nEvent: Caught in or compressed by equipment or objects, unspeciﬁed \nNature: multiple injuries\nFinal Narrative Hospitalized AmputationPart Of\nBodySource\nAn employee's leg was pinned between a truck and\nthe powered pallet jack being operated. The\nemployee was hospitalized for treatment/surgery at\nNavicent Health.1 0 legs pallet jack-\npowered\nPage 1 of 1"

In [ ]:
import openai
from datetime import datetime
openai.api_key = os.environ['OPENAI_API_KEY']  # Replace with your API key

def ask_gpt3(content,question):

    prompt = f"Please answer the following question based on the information provided below. If the answer is not found in the content, please reply with 'don't know'.\n\n{content}\n\nQuestion: {question}"
    response = openai.Completion.create(
        engine="davinci",
        prompt=prompt,
        max_tokens=250,
        n=1,
        stop=None,
        temperature=0.5
    )

    answer = response.choices[0].text.strip()
    return answer
t1 = datetime.now()
answer = ask_gpt3(docs[0].page_content,query)
answer = answer.split("\n")[0]
answer = answer.replace("Answer:","").strip()
t2 = datetime.now()
print(answer,answer.split("\n")[0],t2-t1)